# NB06 CLEAN — Musanga satellite abundance and spatial structure (V3.1)

**New reproducible workflow; previous 83-cell NB06 is archived unchanged.** Goal: combine Sentinel-2 seasonal signals (S), 64D annual embeddings (E), and native 4.77 m PlanetScope structure (P) to predict UAV-derived Musanga canopy fraction.

**Research sequence**
1. Reproduce S2-only, E-only, S2+E 50 m baseline from frozen reference cells, forest- and abundance-balanced; distinguish 400 m buffered spatial folds (where R² ~0.6 was obtained) from **strict LOFO**, which was considerably harder.
2. Test spatial context of 64D embeddings with a genuine **2D CNN**, matched against an embedding pointwise baseline on *identical held-out 50 m cells*.
3. Train the **actual UAV-style ResNet18 U-Net** with Planet B/G/R/NIR native 4.77 m patches, comparing train-only global scaling, per-tile scaling and dual reflectance. Model changes documented, not silently substituting the previous small Planet U-Net.
4. Evaluate aggregation onto **common** 10, 30, 50, 75, 100, 125 m grids, not old tile-local cells.
5. Evaluate learned S/E/P branches and all seven input combinations on the same cells/folds, with high-cover bias/recall, equal-forest MAE/R² and calibration.

**Immutable rules:** Baego, MOS (both mosaics held out together), Yangambi, Yoko. Four independent held-out forests. No held-out label leakage; strict reference & footprint comparisons. UAV maps are pseudo-labels, not independent ground truth. Threshold for high cover = 40%. Each stage saves its own inputs, manifests, predictions, models and validation tables.

**Status:** Step 1 runnable below; Steps 2–5 are locked experiment protocols, to be implemented after the baseline is verified. This deliberate division avoids another sprawling experimental notebook.


## 00 — Setup and discover exact satellite baseline sample

The **Section 10 authoritative 14,454-cell UAV target table** defines the sample support. Cached combined satellite inputs must match its IDs and target fractions exactly. Archived 11B LOFO prediction files are optional checks, not required inputs. No full-mosaic processing.


In [ ]:
# 00 — DRIVE must already be mounted
from pathlib import Path
import numpy as np,pandas as pd,os,json,hashlib
from IPython.display import display
BASE=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3_1/06_spatial_planet_embeddings')
B11=BASE/'11_spatial_ablation_50m'; OUT=BASE/'CLEAN_NB06';OUT.mkdir(parents=True,exist_ok=True)
SITES=('Baego','MOS','Yangambi','Yoko');KEYS=['site','mosaic','row','col'];HIGH=.4
# Preferred independent reference: authoritative Section 10 frozen accepted-cell table.
# Archived 11B model predictions are OPTIONAL; no longer a blocking dependency.
spatial_file=BASE/'10_full_spatial_features_50m.parquet'
assert spatial_file.is_file(),('Missing authoritative Section 10 target table',str(spatial_file))
reference=pd.read_parquet(spatial_file)
assert len(reference)==14454 and not reference.duplicated(KEYS).any()
assert 'target_fraction' in reference.columns and 'accepted' in reference.columns
assert reference.accepted.astype(bool).all()
reference=reference.sort_values(KEYS).reset_index(drop=True)
print('Authoritative 50m reference:',spatial_file,'cells',len(reference))
ref={}
for site in SITES:
    matches=sorted((B11/'fold_predictions').glob(
        f'*/LEAVE_FOREST_OUT_{site}_B0_S2_EMB.parquet'))
    if len(matches)==1:
        z=pd.read_parquet(matches[0],columns=KEYS+['observed','predicted'])
        ref[site]=z.sort_values(KEYS).reset_index(drop=True)
    else:
        print('Archive B0 LOFO file not required:',site,'found',len(matches),
              '— checking Section 10 reference instead')
candidates=sorted((B11/'inputs').glob('combined_50m_*.parquet'));found=[]
for path in candidates:
    d=pd.read_parquet(path)
    if d.duplicated(KEYS).any() or 'target_fraction' not in d:continue
    satellite=[c for c in d if c.startswith(('RAW_ANNUAL__','S2_QUARTERLY_3YR__','EMB_ANNUAL__'))]
    planet=[f'PLANET_{q}_{b}' for q in ('mean','std') for b in ('B','G','R','N')]
    planet += [f'PLANET_texture_{b}' for b in ('NIR_CV_r1','NIR_CV_r3','NIR_CV_r7','NIR_highpass')]
    context=[f'EMB_{q}_{j:02d}' for q in ('mean','sd') for j in range(1,65)]
    count=[f'S2_QUARTERLY_3YR__{i:02d}' for i in (11,22,33,44)]
    required=satellite+planet+context
    if not set(required+count).issubset(d.columns):continue
    good=np.isfinite(d[required].to_numpy('float32')).all(1)&(d[count].to_numpy('float32')>=1).all(1)&d.target_fraction.between(0,1).to_numpy()
    d=d.loc[good].sort_values(KEYS).reset_index(drop=True)
    # The authoritative reference does not depend on completed model fits.
    agree=(len(d)==len(reference) and d[KEYS].equals(reference[KEYS])
           and np.allclose(d.target_fraction,reference.target_fraction,rtol=0,atol=1e-6))
    # If available, also cross-validate the original held-out predictions.
    if agree:
        for forest,b in ref.items():
            a=d.loc[d.site==forest,KEYS+['target_fraction']].reset_index(drop=True)
            if len(a)!=len(b) or not a[KEYS].equals(b[KEYS]) or not np.allclose(
                    a.target_fraction,b.observed,rtol=0,atol=1e-6):
                agree=False
                break
    if agree:found.append((path,d))
assert len(found)==1,('Expected unique 11A reference', [str(p) for p,_ in found])
SOURCE,D=found[0]
assert len(D)==14454 and {'x','y'}.issubset(D.columns)
FEATURES={'S2':[c for c in D if c.startswith(('RAW_ANNUAL__','S2_QUARTERLY_3YR__'))],
          'EMB':[c for c in D if c.startswith('EMB_ANNUAL__')]}
assert len(FEATURES['EMB'])==64
FEATURES['S2_EMB']=FEATURES['S2']+FEATURES['EMB']
D.to_parquet(OUT/'frozen_50m_reference.parquet',index=False)
print('Frozen',len(D),'matched 50m cells; source:',SOURCE)
print('Predictors:',{k:len(v) for k,v in FEATURES.items()})
display(D.groupby('site').size())



## 01 — Reproducible balanced satellite baseline (S2, E, S2+E)

Use identical 50 m targets and features, three spatial-block folds (400 m, 150 m buffer) **and** four LOFO splits. HGB five-member ensembles, train cap 1500 per forest, target 25% high Musanga ≥40% per selected training forest when available, plus balanced draws among remaining abundance bins. All held-out cells retained. **Do not pool MOS-dominated observations for model selection.**

**Baseline provenance note:** previous ~0.6 spatial R² cannot be assumed for this exact reimplementation. Reproduction is judged by new output, not hard-coded scores.


In [ ]:
# 01A — splits and balanced training selection (no held-out labels in sampling)
from scipy.spatial import cKDTree
from sklearn.ensemble import HistGradientBoostingRegressor
from threadpoolctl import threadpool_limits
threadpool_limits(1)
SEED=20261026;CAP=1500;MEMBERS=5
MAX_NEW_FITS=3   # Smoke-test. Switch to None to complete 105 fits.
PARAM=dict(learning_rate=.06,max_iter=180,max_leaf_nodes=20,min_samples_leaf=35,
           l2_regularization=2,early_stopping=False)
CFG=dict(version='NB06_CLEAN_S2EMB_v1',source=str(SOURCE),n=len(D),seed=SEED,
         cap=CAP,members=MEMBERS,model=PARAM,feature_lengths={k:len(v) for k,v in FEATURES.items()})
RUN=OUT/'01_satellite'/hashlib.sha256(json.dumps(CFG,sort_keys=True).encode()).hexdigest()[:16]
RUN.mkdir(parents=True,exist_ok=True)
cf=RUN/'config.json'
if cf.exists():assert json.loads(cf.read_text())==CFG
else:cf.write_text(json.dumps(CFG,indent=2))
y=D.target_fraction.to_numpy(float);site=D.site.to_numpy();xy=D[['x','y']].to_numpy(float)
bx=np.floor(xy[:,0]/400).astype('int64');by=np.floor(xy[:,1]/400).astype('int64')
fold=np.mod(bx*73856093+by*19349663,3)
FOLDS=[('SPATIAL',str(i)) for i in range(3)]+[('LOFO',s) for s in SITES]
def split(kind,hold):
    test=(fold==int(hold)) if kind=='SPATIAL' else (site==hold)
    allowed=~test
    if kind=='SPATIAL':
        for forest in SITES:
            it=np.flatnonzero(test&(site==forest));tr=np.flatnonzero(allowed&(site==forest))
            if len(it) and len(tr):
                near=cKDTree(xy[it]).query(xy[tr],k=1)[0]<150
                allowed[tr[near]]=False
    if kind=='LOFO':assert not (site[allowed]==hold).any()
    return np.flatnonzero(allowed),np.flatnonzero(test)
def draw_balanced(pool,seed):
    rng=np.random.default_rng(seed);selected=[]
    for forest in SITES:
        ix=pool[site[pool]==forest]
        if not len(ix):continue
        cap=min(CAP,len(ix))
        if len(ix)<=cap:selected.extend(ix.tolist());continue
        labels=np.digitize(y[ix],[.10,.20,.40]);bins={b:ix[labels==b] for b in range(4)}
        n=min(len(bins[3]),int(np.ceil(cap*.25)))
        pick=rng.choice(bins[3],n,replace=False).tolist() if n else []
        bins[3]=np.setdiff1d(bins[3],pick)
        while len(pick)<cap:
            active=[b for b in bins if len(bins[b])]
            assert active
            for b in active:
                n=min(len(bins[b]),max(1,(cap-len(pick))//len(active)),cap-len(pick))
                if n:
                    take=rng.choice(bins[b],n,replace=False)
                    pick.extend(take.tolist());bins[b]=np.setdiff1d(bins[b],take)
                if len(pick)==cap:break
        selected.extend(pick)
    return np.array(sorted(selected),dtype=int)
print('Ready:',len(FOLDS),'validation folds',len(FEATURES),'predictor families')


In [ ]:
# 01B — checkpointed model fits, fixed fold membership and held-out data
jobs=[(kind,hold,family,m) for kind,hold in FOLDS for family in FEATURES for m in range(MEMBERS)]
new=0
for kind,hold,family,m in jobs:
    train,test=split(kind,hold)
    f=RUN/f'{kind}_{hold}_{family}_m{m}.npz'
    if f.is_file():
        with np.load(f,allow_pickle=False) as z:
            assert np.array_equal(z['test_index'],test) and len(z['pred'])==len(test)
        continue
    if MAX_NEW_FITS is not None and new>=MAX_NEW_FITS:break
    indices=draw_balanced(train,SEED+107*m+sum(map(ord,hold))+len(family)*13)
    X=D[FEATURES[family]].to_numpy('float32')
    estimator=HistGradientBoostingRegressor(random_state=SEED+m,**PARAM)
    estimator.fit(X[indices],y[indices]);pred=np.clip(estimator.predict(X[test]),0,1)
    with open(f.with_suffix('.partial'),'wb') as dest:
        np.savez_compressed(dest,test_index=test,pred=pred.astype('float32'))
    os.replace(f.with_suffix('.partial'),f)
    new+=1;print('Finished',kind,hold,family,'member',m,flush=True)
print('Completed',len(list(RUN.glob('*.npz'))),'of',len(jobs),'fits in',RUN)


In [ ]:
# 01C — final metrics ONLY after all 5 ensemble members exist per case
out=[]
for kind,hold in FOLDS:
    train,test=split(kind,hold)
    for family in FEATURES:
        files=[RUN/f'{kind}_{hold}_{family}_m{m}.npz' for m in range(MEMBERS)]
        if not all(f.exists() for f in files):continue
        preds=[]
        for f in files:
            with np.load(f,allow_pickle=False) as z:
                assert np.array_equal(z['test_index'],test)
                preds.append(z['pred'].astype(float))
        prediction=np.mean(preds,axis=0)
        for forest in SITES:
            mask=site[test]==forest
            if not mask.any():continue
            a=y[test][mask];b=prediction[mask];e=b-a;high=a>=HIGH
            out.append(dict(scheme=kind,holdout=hold,model=family,forest=forest,n=len(a),
                 MAE_pp=100*np.mean(np.abs(e)),R2=1-np.sum(e**2)/np.sum((a-a.mean())**2) if np.var(a)>0 else np.nan,
                 bias_pp=100*e.mean(),high_n=int(high.sum()),
                 high_bias_pp=100*e[high].mean() if high.any() else np.nan,
                 high_recall=np.mean(b[high]>=HIGH) if high.any() else np.nan,
                 mean_target=a.mean(),target_var=a.var(),mse=np.mean(e*e)))
RES=pd.DataFrame(out)
assert len(RES)>0,'No complete ensemble yet; continue 01B with MAX_NEW_FITS=None'
print('Per-forest results; LOFO contains exactly one held-out forest each:')
display(RES.round(3))
rows=[]
for (scheme,hold,model),g in RES.groupby(['scheme','holdout','model']):
    u=g.mean_target.mean();var=np.mean(g.target_var+(g.mean_target-u)**2)
    rows.append(dict(scheme=scheme,holdout=hold,model=model,n_forests=len(g),
                     equal_forest_MAE_pp=g.MAE_pp.mean(),equal_forest_R2=1-g.mse.mean()/var,
                     mean_forest_R2=g.R2.mean()))
SUMMARY=pd.DataFrame(rows)
display(SUMMARY.round(3))
RES.to_csv(RUN/'validation_by_forest.csv',index=False)
SUMMARY.to_csv(RUN/'validation_summary.csv',index=False)
print('Saved',RUN)


## 02 — Embedding spatial context (next implementation gate)

**Fixed test:** pointwise annual 64D embeddings (Stage 1) vs genuine 2D patches (e.g. 7×7 cells on the *native 10 m embedding lattice*) through a compact CNN; same frozen 50 m cell centers and LOFO/spatial folds. The 2D model must use true geographic neighborhood placement, valid-data masks, **train-forest-only normalization**, and no target leakage. Compare E, E+neighborhood summary, E+2D CNN, SE.

**Do not merely resize 10 m embeddings to 4.77 m and claim higher spatial resolution.** Report improvement on exactly paired test cells and the ability to identify high-cover MOS.


## 03 — PlanetScope 4.77 m ResNet18 U-Net (next implementation gate)

Use 422 verified 64×64 native Planet B/G/R/NIR tiles and dense UAV teacher maps. **Recover the exact UAV ResNet18 U-Net encoder and decoder configuration** from the successful UAV notebook/checkpoint before implementation; the previous small Planet U-Net is not the same model. Adapt the input stem to 4 bands while documenting initialization.

Controlled input treatments: robust global train-forest normalization; tile-relative robust normalization; dual absolute+relative bands. Moderate abundance-balanced, forest-balanced tile sampling; masked dense regression; strict four LOFO. Compare 4.77 m discrimination and high-abundance performance. Keep Planet annual only.


## 04 — Aggregation on common geographic support (next implementation gate)

**Evaluate rather than retrain first:** reproject dense 4.77 m Planet predictions, S2 predictions and embedding predictions onto identical predeclared 10/30/50/75/100/125 m grids. Use area-weighted aggregation, explicit coverage fractions and one exact reference UAV mask for all candidates. Exclude unsupported edge cells consistently. No tile-local 50 m comparison to S2's different grid.

Report per forest and equal forest: R², MAE, RMSE, bias, Pearson r, high-cover bias and recall, and balanced scatterplots.


## 05 — Learned S/E/P fusion and seven ablations (final stage)

After each independent component has passed its validation gate, test S, E, P, SE, SP, EP, SEP under **the same four LOFO held-out forests, identical geographic footprints and reference targets**. S2 supplies seasonality, embeddings supply 2D context, Planet supplies 4.77 m canopy structure. Feature fusion weights are learned; avoid forcing hand-picked contributions. Select on equal-forest accuracy AND MOS high-cover recall/bias, not just Yangambi's low prevalence. Four forests are four independent transfer tests, not thousands of independent field sites.

**Do not add experimental cells above these locked stage contracts.** Implement one self-contained, cached stage at a time after the previous stage is verified.
